# {{EXPERIMENT}} · LIBRARY: image processing (1 of 2)

<!-- describe the experiment here -->

## Configure

Everything specific to this experiment. A scan list left empty skips its section.

In [ ]:
from pathlib import Path

EXPERIMENT = "{{EXPERIMENT}}"
LIBRARY = "LIBRARY"
EXPERIMENT_DATE = "{{DATE}}"

# raw and stitched images on the T7; everything else stays in this folder
DRIVE = Path("{{DRIVE}}")

# the camera: rotation vs the stage, and the offset: a measured dark image (one camera tile; preferred) or a scalar
# "offset". Setup 7 Kinetix: rotation -2.15 deg, dark image from MBO-053 (copy it to 0_config)
CAMERA = {"name": "Kinetix (Setup 7)", "rotation_deg": -2.15, "dark": "0_config/dark_setup7_kinetix.tif"}
MAX_CROSS_PX = 25.0   # how far apart neighbors may sit across a seam (Kinetix at -2.15 deg: ~20 px; centering absorbs it)

# expression time course: scan folder (under 1_raw-images/<EXPRESSION_DIR>) -> minutes after expression started.
# Snapshots, one scan per time point; the dumbbell grid is placed on the last one
EXPRESSION_DIR = "GFP/expression-time-course"
EXPRESSION = {
    # "gfp_20min_kinetix_500ms": 20,
    # "gfp_60min_kinetix_500ms": 60,
}

# GFP on the buttons: the background scan and the SDS washes (folder under 1_raw-images -> label). The buttons are
# found from the reaction chambers and centered on the last wash (GFP left on the buttons only)
GFP_BACKGROUND = ("GFP", "gfp_bg_kinetix_500ms")
SDS_WASHES = {
    # "GFP/sds-wash-series/sds_wash_series_post_SDS_wash4_kinetix_500ms": "SDS wash 4",
}

# NADH standards, per camera: camera -> {folder under 1_raw-images/Standards/<camera>: [NADH] (µM)}
STANDARDS = {
    # "kinetix": {"NADH_standard_0uM_500ms_kinetix": 0, "NADH_standard_1mM_500ms_kinetix": 1000},
}

# assay runs: folder under 1_raw-images/<ASSAY_DIR> -> [S] (µM); the chamber grid is placed on the highest [S]
ASSAY_DIR = "Assays"
ASSAY_RUNS = {
    # "NADH_kinetics_5mM_BzP_kinetix_500ms": 5000,
}
SUBSTRATE = "bzp_uM"
TAG_SUFFIX = None   # how figures write a TAG ("*") substitution: None "A4tag"; the amino acid incorporated, e.g. "AHA" (azidohomoalanine): "A4AHA"

# the scan whose seams every scan reuses (bright and textured), and the GFP scans' flatfield (BaSiC, fit once)
SEAM_SCAN = None        # a key of the scans below, e.g. the last expression scan; None: the first scan stitched
FLATFIELD_SCAN = None   # the same; None: SEAM_SCAN

# the chip
N_ROWS, N_COLS = 56, 32
PINLIST_FILE = "{{PINLIST}}"   # Array-Print's pinlist: one copy per print in ../_config/_pinlists
MIRROR_COLS = True                     # the print's frame is the image mirrored (MBO-040 on)

# masks (Kinetix px; Zyla ~0.55x): the reaction and DNA chambers (placed as two grids; the neck joins each pair), the
# button, the chamber ROI
REACTION_R, DNA_R = 90, 65   # circle radii
DNA_OFFSET = (234, 0)        # the DNA grid's first placement: the reaction grid moved by this (px)
NECK_WIDTH = 36              # the neck between each chamber's two circles (px)
DUMBBELL_BG = 18             # the dumbbell's background ring: this far beyond its outline (px)
BUTTON_R, BUTTON_RING = 34, (38, 52)      # ring inside the button rim
CHAMBER_R, CHAMBER_RING = 65, (95, 115)   # ring outside the chamber; never subtracted
CENTER_REACH = 30                         # chamber centering search around the hand grid (px)
BUTTON_REACH = 30                         # button centering search around the reaction chamber + offset (px)
BUTTON_SPOT_R = 20                        # the GFP spot on a button (px): the disc buttons are centered with
SEED_CORNERS = None   # a first placement for the first grid (e.g. another chip's corners JSON, copied to 0_config)

# the grid windows: None opens each until its corners file exists; True / False: always / never
PLACE_DUMBBELLS = PLACE_CHAMBERS = None

## Setup

The experiment from the settings above: its folders, pinlist and scans (`hp.Experiment`). Each step below is one call; the code is in `htmek_processing/experiment.py`.

In [ ]:
import htmek_processing as hp

exp = hp.Experiment.from_config(globals())

# 1. Stitch

The seam scan first (its seams measured, and the GFP flatfield fit on `FLATFIELD_SCAN`), then every scan reusing both.
Existing stores are reopened, not restitched.

QC: every tile corner, on the seam scan.

In [ ]:
exp.stitch_all()

**Tile check:** every tile's median on every scan (every pass of an assay) against its usual share of the scan; a tile darker or brighter than usual on one scan is flagged, and the analysis notebook leaves out the buttons and chambers in it.

In [ ]:
exp.check_tiles()

# 2. Expression time course

### 2a. Reaction- and DNA-chamber grids (interactive), on the time course's last scan

Two grids in one window: the reaction chambers and the DNA chambers, each with its own corners and local corrections; Tab
switches which one you drag, and each chamber's neck is drawn between its two circles. Saves
`2_masks/reaction_chamber_corners.json` / `_warp.json` (with the neck and the background margin) and
`2_masks/dna_chamber_corners.json` / `_warp.json`. The window blocks until it is closed.

In [ ]:
exp.place_dumbbells()

### 2b. Center the reaction and DNA chambers

Each circle moves from the placed grid to where a disc of its radius holds the most signal on the last expression scan,
within `CENTER_REACH` px (`hp.center_on_disc_sum`). A fit counts only where the chamber stands out from the ring around it
(above the BLANK chambers' median + 3 SD of that contrast); the rest, chambers that did not express, take their
neighbors' shift (`hp.shift_from_neighbors`). The grid's 4 corners can't follow the tile-to-tile jumps the
stitch leaves (up to ~20 px across a seam); this does. Writes `2_masks/reaction_chamber_centers.csv` and
`dna_chamber_centers.csv`. QC: examples, the grid dashed and the centered circle solid.

In [ ]:
exp.center_dumbbells()

### 2c. GFP per dumbbell at every time point

The dumbbell's sum and mean (each chamber's reaction chamber, its DNA chamber and the neck between them), and the sum
minus a ring around its outline (`hp.dumbbell_intensities`), on every scan of the time course. Writes `4_exports/expression/gfp_expression_per_dumbbell.csv`. Uses the centered reaction and DNA chambers. QC: the reaction chambers on the last scan,
and the controls' GFP over time.

In [ ]:
expression = exp.quantify_expression()

# 3. Buttons

### 3a. Buttons from the reaction chambers, centered on the last SDS wash

Each button sits a fixed offset from its reaction chamber: the offset that puts the most last-wash signal under
button-sized discs around the centered reaction chambers (searched ±120 px, then 1 px). Buttons with a GFP spot
(the spot's median minus the ring around it, above the BLANK buttons' median + 3 SD) are then centered on it with a disc the spot's size
(`BUTTON_SPOT_R`, within `BUTTON_REACH`; `hp.center_on_disc_sum`); the rest take their neighbors' shift. Writes `2_masks/button_centers.csv`. QC: examples.

QC: the button ROIs on the last wash, and every button as a crop laid out as the chip (variant, r/c and bg-subtracted button sum on each; one contrast; one PDF page).

In [ ]:
exp.find_buttons()

### 3b. GFP per button

Every GFP scan (time course, background, washes): the button sum and median, the ring's median, and the button sum
minus the same button on the background scan. Writes `4_exports/gfp/gfp_per_chamber.csv`.

In [ ]:
gfp = exp.quantify_buttons()

# 4. Assay

### 4a. Chamber grid (interactive), on the highest [S] run's last pass

Saves `2_masks/assay_chamber_corners.json` and `_warp.json`; the ROIs are then centered on each chamber's shape (the most
signal under a disc of `CHAMBER_R`, within `CENTER_REACH`), and every run uses the same masks.

In [ ]:
exp.place_chambers()

### 4b. NADH per chamber at every pass

The chamber ROI median (RFU) at every pass of every run, each chamber timed by its own tile. Every pass is kept, flagged
`complete_pass` (every tile imaged) and `tile_imaged`; the analysis notebook filters. Writes
`4_exports/assay/nadh_per_chamber.csv`.

In [ ]:
nadh = exp.quantify_assay()

# 5. NADH standards

Each camera's standards read with the assay's chamber masks and ROI, no flatfield. The calibration (a line per chamber) is
fit in the analysis notebook. Writes `4_exports/assay/standards/<camera>/nadh_standards_per_chamber.csv`.

In [ ]:
standards = exp.quantify_standards()